# Phase 1: Download + Load Data & Phase 2: Data Audit

This notebook covers the EDA process for the IEEE-CIS Fraud Detection dataset.

In [ ]:
!pip install -q kaggle python-dotenv

import os
from dotenv import load_dotenv

load_dotenv('../.env')

os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME', '')
os.environ['KAGGLE_KEY'] = os.getenv('KAGGLE_KEY', '')

if not os.path.exists('../data/raw/train_transaction.csv'):
    !kaggle competitions download -c ieee-fraud-detection -p ../data/raw
    !unzip -q -o ../data/raw/ieee-fraud-detection.zip -d ../data/raw
    !rm ../data/raw/ieee-fraud-detection.zip
else:
    print('Dataset already downloaded.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Read data (Phase A: 100k sample rows as per strategy)
train_transaction = pd.read_csv('../data/raw/train_transaction.csv', nrows=100000)
train_identity = pd.read_csv('../data/raw/train_identity.csv')

print(f'Transaction shape: {train_transaction.shape}')
print(f'Identity shape: {train_identity.shape}')

In [ ]:
# Merge datasets
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
print(f'Merged shape: {train.shape}')

## Phase 2: Data Audit

In [ ]:
# Target distribution
print(train['isFraud'].value_counts(normalize=True))

In [ ]:
# Missing values
missing_pct = train.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 85].index.tolist()
print(f'Columns with >85% missing values: {len(cols_to_drop)}')

# Save columns strategy to a file or list
columns_strategy = {
    'drop': cols_to_drop
}
import json
with open('../data/processed/columns_strategy.json', 'w') as f:
    json.dump(columns_strategy, f)